# 🛡️ Chirp AI — Modelo de toxicidad (ESPAÑOL)
### Proyecto Final · Inteligencia Artificial y MLOps (UCB)

Entrena un clasificador de toxicidad en español (TF-IDF + Regresión Logística) con el dataset **Spanish Hate Speech Superset** (~30.000 posts) y exporta `toxicity_model.pkl` para el microservicio de Chirp.

### ✅ Antes de empezar (una sola vez)
1. Cuenta gratis en **huggingface.co**.
2. Entra a **huggingface.co/datasets/manueltonneau/spanish-hate-speech-superset** y pide/acepta el acceso.
3. Crea un token en **Settings → Access Tokens** (tipo *Read*).
4. Pega ese token en la celda 2 (donde dice `HF_TOKEN`).

Luego: **Entorno de ejecución → Ejecutar todo**.

## 1. Instalar dependencias

In [ ]:
!pip install -q scikit-learn pandas numpy matplotlib seaborn joblib datasets huggingface_hub
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns

## 2. Token y carga del dataset

👉 **Pega tu token de Hugging Face** entre las comillas de `HF_TOKEN`.

In [ ]:
# ▼▼▼ PEGA TU TOKEN AQUÍ ▼▼▼
HF_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
# ▲▲▲ PEGA TU TOKEN AQUÍ ▲▲▲

from huggingface_hub import login
login(token=HF_TOKEN)

from datasets import load_dataset
ds = load_dataset("manueltonneau/spanish-hate-speech-superset", split="train")

df = ds.to_pandas().rename(columns={"labels": "toxic"})[["text", "toxic"]]
df["text"] = df["text"].astype(str)
df["toxic"] = (df["toxic"] > 0).astype(int)
print("Filas cargadas:", df.shape)
df.head()

## 3. Control de calidad y EDA

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
df = df[df['text'].str.strip()!=''].reset_index(drop=True)
print('Filas:', len(df))
print('Balance de clases:\n', df['toxic'].value_counts())
df['len']=df['text'].str.len()
fig,ax=plt.subplots(1,2,figsize=(12,4))
df['toxic'].value_counts().sort_index().plot(kind='bar',ax=ax[0],color=['#1d9bf0','#e0245e'])
ax[0].set_title('0 = no toxico | 1 = toxico'); ax[0].set_xticklabels(['no toxico','toxico'],rotation=0)
sns.boxplot(data=df,x='toxic',y='len',ax=ax[1]); ax[1].set_title('Longitud del texto por clase')
plt.tight_layout(); plt.show()

## 4. Entrenamiento (TF-IDF + Regresión Logística)

⚠️ Los pasos del Pipeline se llaman **'tfidf'** y **'clf'** — así los espera el microservicio para resaltar palabras. No cambies esos nombres.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, f1_score

Xtr,Xte,ytr,yte = train_test_split(df['text'], df['toxic'], test_size=0.2,
                                   random_state=42, stratify=df['toxic'])
pipe = Pipeline([('tfidf', TfidfVectorizer(ngram_range=(1,2), min_df=2, sublinear_tf=True)),
                 ('clf',   LogisticRegression(max_iter=1000, C=4.0, class_weight='balanced'))])
pipe.fit(Xtr, ytr)
pred = pipe.predict(Xte)
print('F1:', round(f1_score(yte,pred),3))
print(classification_report(yte, pred, target_names=['no toxico','toxico']))

## 5. Evaluación (matriz de confusión)

In [ ]:
cm=confusion_matrix(yte,pred)
plt.figure(figsize=(4,3))
sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=['ok','toxico'],yticklabels=['ok','toxico'])
plt.ylabel('real'); plt.xlabel('predicho'); plt.title('Matriz de confusion'); plt.show()

## 6. Palabras más tóxicas (interpretación)

In [ ]:
vocab=np.array(pipe.named_steps['tfidf'].get_feature_names_out())
coef=pipe.named_steps['clf'].coef_[0]
print('TOP toxicas:', list(vocab[np.argsort(coef)[-20:]][::-1]))
print('\nTOP no toxicas:', list(vocab[np.argsort(coef)[:15]]))

## 7. Probar el modelo

In [ ]:
for t in ['gracias por el gran trabajo equipo','eres un idiota, callate']:
    p=pipe.predict_proba([t])[0][1]
    print(f'  toxicidad={p:.0%}  |  {t}')

## 8. Exportar y descargar el modelo

Se descarga `toxicity_model.pkl`. Cópialo a `ai-service/models/toxicity_model.pkl` de tu proyecto y haz `docker compose restart ai-service`.

In [ ]:
import joblib
joblib.dump(pipe, 'toxicity_model.pkl')
from google.colab import files
files.download('toxicity_model.pkl')
print('Listo. Coloca el archivo en ai-service/models/ de tu proyecto.')